In [7]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.7/207.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 713.6/713.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.7 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00


In [17]:
#=============================================
# Real-Time Weather Information (Open Meteo)
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup API client
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# API parameters (Pune location)
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 18.5196,
    "longitude": 73.8554,
    "daily": "weather_code",
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m"],
    "timezone": "auto",
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

# Process hourly data
hourly = response.Hourly()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),
    "temperature": hourly.Variables(0).ValuesAsNumpy(),
    "humidity": hourly.Variables(1).ValuesAsNumpy(),
    "wind_speed": hourly.Variables(2).ValuesAsNumpy()
}

df = pd.DataFrame(hourly_data)

# -------- MENU SYSTEM --------
while True:
    print("\n🌦️ WEATHER DASHBOARD (PUNE)")
    print("1. View Top 10 Records")
    print("2. View Summary Statistics")
    print("3. Filter Hot Hours (Temp > 30°C)")
    print("4. Save Data to CSV")
    print("5. Exit")

    choice = input("\nEnter your choice: ")

    if choice == "1":
        print("\n📊 Top 10 Records:")
        print(df.head(10).to_string(index=False))

    elif choice == "2":
        print("\n📈 Summary Statistics:")
        print(df.describe())

    elif choice == "3":
        hot = df[df["temperature"] > 30]
        print(f"\n🔥 Hot Hours Found: {len(hot)}")
        print(hot[["date", "temperature"]].head(10))

    elif choice == "4":
        df.to_csv("weather_data.csv", index=False)
        print("\n💾 Data saved as 'weather_data.csv'")

    elif choice == "5":
        print("\n👋 Exiting... Thank you!")
        break

    else:
        print("\n❌ Invalid choice. Try again.")


🌦️ WEATHER DASHBOARD (PUNE)
1. View Top 10 Records
2. View Summary Statistics
3. Filter Hot Hours (Temp > 30°C)
4. Save Data to CSV
5. Exit

Enter your choice: 1

📊 Top 10 Records:
                     date  temperature  humidity  wind_speed
2026-03-26 00:00:00+00:00    26.114500      76.0    3.600000
2026-03-26 01:00:00+00:00    25.564499      80.0    3.877318
2026-03-26 02:00:00+00:00    25.264500      83.0    4.198285
2026-03-26 03:00:00+00:00    24.764500      86.0    3.600000
2026-03-26 04:00:00+00:00    24.164499      89.0    3.219938
2026-03-26 05:00:00+00:00    23.814499      86.0    2.545584
2026-03-26 06:00:00+00:00    23.614500      86.0    2.414953
2026-03-26 07:00:00+00:00    24.114500      81.0    2.160000
2026-03-26 08:00:00+00:00    28.614500      58.0    0.720000
2026-03-26 09:00:00+00:00    31.564499      43.0    6.369050

🌦️ WEATHER DASHBOARD (PUNE)
1. View Top 10 Records
2. View Summary Statistics
3. Filter Hot Hours (Temp > 30°C)
4. Save Data to CSV
5. Exit

Enter